In [ ]:
import numpy as np

# Each row = one student
# Column 1 = CGPA (normalized: divide by 10)
# Column 2 = Coding problems solved (normalized: divide by 100)

#              CGPA   Problems   →   Placed?
# Student 1:   6.0    20             No  (low cgpa, few problems)
# Student 2:   8.5    80             Yes (good cgpa, many problems)
# Student 3:   9.0    15             No  (great cgpa but no coding practice)
# Student 4:   7.0    90             Yes (decent cgpa, lots of practice)
# Student 5:   5.5    10             No  (weak on both)
# Student 6:   8.0    70             Yes (strong on both)
# Student 7:   7.5    60             Yes (balanced)
# Student 8:   6.5    25             No  (below average both)

# Normalize inputs to range 0–1 (neural networks learn better this way)
# Input
X = np.array([
    [6.0/10, 20/100],   # Student 1
    [8.5/10, 80/100],   # Student 2
    [9.0/10, 15/100],   # Student 3
    [7.0/10, 90/100],   # Student 4
    [5.5/10, 10/100],   # Student 5
    [8.0/10, 70/100],   # Student 6
    [7.5/10, 60/100],   # Student 7
    [6.5/10, 25/100],   # Student 8
])

# Ouput
y = np.array([[0],[1],[0],[1],[0],[1],[1],[0]])  # 0=Not placed, 1=Placed

print("Input shape:", X.shape)   # (8, 2) — 8 students, 2 features
# print("Input shape:\n", X)
print("Label shape:", y.shape)   # (8, 1) — 8 labels
# print("\nFirst student — CGPA:6.0, Problems:20 → Placed:", y[0][0])
# print("Second student — CGPA:8.5, Problems:80 → Placed:", y[1][0])

Input shape: (8, 2)
Input shape: [[0.6  0.2 ]
 [0.85 0.8 ]
 [0.9  0.15]
 [0.7  0.9 ]
 [0.55 0.1 ]
 [0.8  0.7 ]
 [0.75 0.6 ]
 [0.65 0.25]]
Label shape: (8, 1)


In [3]:
# ── ACTIVATION FUNCTIONS ──────────────────────────────────────
def sigmoid(x):
    # Converts any number → 0 to 1
    # Perfect for "probability of getting placed"
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def relu(x):
    # Hidden layers use ReLU — it's faster to train
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


In [ ]:
# ── THE NETWORK ───────────────────────────────────────────────
class PlacementPredictor:
    """
    Architecture:
      Input layer  : 2 neurons (CGPA, Problems solved)
      Hidden layer : 4 neurons (learns patterns like "needs both skills")
      Output layer : 1 neuron  (probability of placement: 0.0 to 1.0)
    """
    def __init__(self):
        np.random.seed(42)
        
        # Layer 1 weights: shape (2, 4)
        # 2 inputs connecting to 4 hidden neurons
        # Think of each hidden neuron as learning one pattern:
        #   Neuron 1 → "Is CGPA high?"
        #   Neuron 2 → "Are problems solved high?"
        #   Neuron 3 → "Is student balanced?"
        #   Neuron 4 → "Is student weak overall?"
        self.W1 = np.random.randn(2, 4) * 0.5
        self.b1 = np.zeros((1, 4))
        # print(f"W1:\n {self.W1}")

        # Layer 2 weights: shape (4, 1)
        # 4 hidden neurons connecting to 1 output neuron
        self.W2 = np.random.randn(4, 1) * 0.5
        self.b2 = np.zeros((1, 1))

    def forward(self, X):
        """
        Forward pass — data flows left to right through the network.
        
        X (8,2) → [W1 (2,4)] → z1 (8,4) → relu → a1 (8,4)
                             → [W2 (4,1)] → z2 (8,1) → sigmoid → output (8,1)
        """
        # Hidden layer
        self.z1 = X @ self.W1 + self.b1   # linear step
        self.a1 = relu(self.z1)            # activation — adds non-linearity

        # Output layer
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = sigmoid(self.z2)         # squash to 0–1 = placement probability

        return self.a2

    def backward(self, X, y_true, lr=0.1):
        """
        Backward pass — we measure the error at the output
        and push the blame backwards through each layer.
        
        This is how the network learns:
          If it predicted 0.9 (placed) but actual = 0 (not placed),
          it adjusts weights to predict lower next time.
        """
        m = len(X)

        # How wrong is the output layer?
        error_output = self.a2 - y_true                         # shape (8,1)
        delta2 = error_output * sigmoid_derivative(self.z2)     # shape (8,1)

        # Gradient for W2 and b2
        dW2 = self.a1.T @ delta2 / m
        db2 = np.mean(delta2, axis=0, keepdims=True)

        # Push error back to hidden layer
        error_hidden = delta2 @ self.W2.T                       # shape (8,4)
        delta1 = error_hidden * relu_derivative(self.z1)        # shape (8,4)

        # Gradient for W1 and b1
        dW1 = X.T @ delta1 / m
        db1 = np.mean(delta1, axis=0, keepdims=True)

        # Update weights — take a small step in the right direction
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1

    def train(self, X, y, epochs=3000, lr=0.1):
        print("Training placement predictor...\n")
        for epoch in range(epochs):
            preds = self.forward(X)
            loss  = mse_loss(y, preds)
            self.backward(X, y, lr)

            if epoch % 500 == 0:
                print(f"  Epoch {epoch:4d} | Loss: {loss:.6f}")

        print("\nTraining complete!")

In [5]:
# ── TRAIN ─────────────────────────────────────────────────────
model = PlacementPredictor()
model.train(X, y, epochs=3000, lr=0.1)

Training placement predictor...

  Epoch    0 | Loss: 0.278384
  Epoch  500 | Loss: 0.134697
  Epoch 1000 | Loss: 0.035305
  Epoch 1500 | Loss: 0.014950
  Epoch 2000 | Loss: 0.008634
  Epoch 2500 | Loss: 0.005816

Training complete!


In [6]:
# ── EVALUATE ON TRAINING DATA ─────────────────────────────────
print("\n" + "="*52)
print(f"{'Student':<10} {'CGPA':<8} {'Problems':<12} {'Prob%':<10} {'Pred':<8} {'Actual'}")
print("="*52)

probs = model.forward(X)
names = ["Arjun","Priya","Ravi","Sneha","Dev","Meera","Karan","Zara"]

for i in range(len(X)):
    cgpa     = X[i][0] * 10        # de-normalize for display
    problems = X[i][1] * 100
    prob     = probs[i][0]
    pred     = 1 if prob >= 0.5 else 0
    actual   = y[i][0]
    status   = "✓" if pred == actual else "✗"

    print(f"  {names[i]:<9} {cgpa:<8.1f} {problems:<12.0f} "
          f"{prob*100:<10.1f} {pred:<8} {actual}  {status}")

accuracy = np.mean((probs >= 0.5).astype(int) == y) * 100
print("="*52)
print(f"\nAccuracy: {accuracy:.1f}%")


Student    CGPA     Problems     Prob%      Pred     Actual
  Arjun     6.0      20           6.3        0        0  ✓
  Priya     8.5      80           98.8       1        1  ✓
  Ravi      9.0      15           3.8        0        0  ✓
  Sneha     7.0      90           99.6       1        1  ✓
  Dev       5.5      10           3.8        0        0  ✓
  Meera     8.0      70           96.0       1        1  ✓
  Karan     7.5      60           88.2       1        1  ✓
  Zara      6.5      25           10.9       0        0  ✓

Accuracy: 100.0%


In [7]:
# ── PREDICT A BRAND NEW STUDENT ───────────────────────────────
print("\n── New student prediction ──")
new_student = np.array([[7.8/10, 65/100]])  # CGPA: 7.8, Problems: 65
result = model.forward(new_student)[0][0]
print(f"CGPA: 7.8 | Problems solved: 65")
print(f"Placement probability: {result*100:.1f}%")
print(f"Prediction: {'PLACED ✓' if result >= 0.5 else 'NOT PLACED ✗'}")


── New student prediction ──
CGPA: 7.8 | Problems solved: 65
Placement probability: 93.1%
Prediction: PLACED ✓
